## SCHEMA VALIDATION & OUTLIER DETECTION

## 🧠 WHAT ARE OUTLIERS? (Very Simple)

Outliers are data points that are unusually far from the majority of the data.

**Example:**
Most car prices are between **12k – 40k**.  
If one row suddenly has:

Sell Price = **2 Lakhs**

That value:
- ❌ Can distort the model  
- ❌ Can skew averages  
- ❌ Can create unstable predictions  

Outliers are **not always wrong**, but they must be **detected and reviewed**.


## 🧠 WHAT IS IQR? (Interquartile Range)

**Definition:**  
IQR measures the spread of the middle 50% of the data.

**How it works:**
- Q1 → 25th percentile  
- Q3 → 75th percentile  
- IQR = Q3 − Q1  

**Outlier rule:**
- Lower bound = Q1 − 1.5 × IQR  
- Upper bound = Q3 + 1.5 × IQR  

Anything outside this range is an outlier.

**Advantages:**
- ✅ Does **NOT** assume normal distribution  
- ✅ Robust to extreme values  

➡️ **Best choice for real ML data**


## 🧠 WHAT IS Z-SCORE?

**Definition (Simple):**  
Z-score tells how many standard deviations a value is away from the mean.

**Formula:**  
Z = (value − mean) / standard deviation

**Rule:**  
|Z| > 3 → outlier

**Limitations:**
- ❌ Sensitive to extreme values  
- ❌ Assumes normal distribution  

➡️ **Useful only when data is clean**


## 🧠 IQR vs Z-Score (MEMORIZE THIS)

| Method  | Use When                     |
|---------|------------------------------|
| IQR     | Real-world, skewed data      |
| Z-Score | Clean, normal data           |

**Interview answer:**

> “I prefer IQR for production ML pipelines.”


# Day 6: Schema Validation & Outlier Detection

Goal:
- Validate incoming ML data
- Detect missing values
- Detect outliers using IQR and Z-score
- Apply QE-style assertions
m

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [19]:
df = pd.read_csv("RAW_DATA\\carprices.csv")
df.head(20)


,Mileage,Age(yrs),Sell Price($)
0,69000,6,18000
1,35000,3,34000
2,57000,5,26100
3,22500,2,40000
4,46000,4,31500
5,59000,5,26750
6,52000,5,32000
7,72000,6,19300
8,91000,8,12000
9,67000,6,22000


## Schema Validation

Schema checks ensure the data structure matches
what the ML model expects.


In [5]:
EXPECTED_COLUMNS = {"Mileage", "Age(yrs)", "Sell Price($)"}

actual_columns = set(df.columns)

missing = EXPECTED_COLUMNS - actual_columns
extra = actual_columns - EXPECTED_COLUMNS

print("Expected columns:", EXPECTED_COLUMNS)
print("Actual columns  :", actual_columns)

if missing:
    print("❌ Missing columns:", missing)
if extra:
    print("⚠️ Extra columns  :", extra)

assert actual_columns == EXPECTED_COLUMNS, "Schema mismatch!"
print("✅ Column schema validation passed")


Expected columns: {'Age(yrs)', 'Sell Price($)', 'Mileage'}
Actual columns  : {'Age(yrs)', 'Sell Price($)', 'Mileage'}
✅ Column schema validation passed


In [7]:
for col in df.columns:
    assert pd.api.types.is_numeric_dtype(df[col].dtype), f"{col} must be numeric"

print("✅ Data type check passed")


✅ Data type check passed


## Missing Value Check

Large amounts of missing data can break ML pipelines.


In [8]:
null_ratio = df.isnull().mean()
print(null_ratio)

assert null_ratio.max() < 0.05, "Too many missing values!"
print("✅ Missing value check passed")


Mileage          0.0
Age(yrs)         0.0
Sell Price($)    0.0
dtype: float64
✅ Missing value check passed


## Range Validation

Catch impossible values early.


In [9]:
assert (df["Mileage"] > 0).all(), "Mileage must be positive"
assert (df["Age(yrs)"] > 0).all(), "Age must be positive"
assert (df["Sell Price($)"] > 0).all(), "Price must be positive"
print("✅ Range checks passed")


✅ Range checks passed


## Outlier Detection using IQR

IQR is robust and preferred for real-world ML datasets.


In [ ]:
Q1 = df["Sell Price($)"].quantile(0.25)
Q3 = df["Sell Price($)"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

iqr_outliers = df[
    (df["Sell Price($)"] < lower) | (df["Sell Price($)"] > upper)
    ]

iqr_outliers


,Mileage,Age(yrs),Sell Price($)


In [11]:
assert len(iqr_outliers) < 0.05 * len(df), "Too many IQR outliers"
print("✅ IQR outlier check passed")


✅ IQR outlier check passed


## Outlier Detection using Z-score

Z-score measures distance from the mean.


In [17]:
z_scores = np.abs(stats.zscore(df["Sell Price($)"]))
zscore_outliers = df[z_scores > 3]

zscore_outliers


,Mileage,Age(yrs),Sell Price($)


In [18]:
assert len(zscore_outliers) < 0.05 * len(df), "Too many Z-score outliers"
print("✅ Z-score outlier check passed")


✅ Z-score outlier check passed


## Important Rule

Outliers are flagged, not blindly removed.

Handling depends on:
- Business logic
- Data source reliability
- Model sensitivity
